# Solutions — DataFrame Thinking: SQL Semantics, NULLs, Partition Pruning, Lazy Plans

Exercise notebook: `12_pyspark_preparation/03_dataframe_thinking.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · A SQL query in plain Python

**Working solution**

In [ ]:
from collections import defaultdict

def country_report(orders):
    groups = defaultdict(lambda: [0, 0.0])
    for o in orders:
        if o.get("status") is None or o["status"] == "CANCELLED":
            continue
        g = groups[o["country"]]
        g[0] += 1
        g[1] += o["amount"]
    rows = [(c, n, round(total, 2)) for c, (n, total) in groups.items() if n >= 2]
    return sorted(rows, key=lambda r: (-r[2], r[0]))

**Explanation**

Reading SQL clause by clause: `WHERE` → filter, `GROUP BY` → accumulator dict, `HAVING` → filter on aggregates, `ORDER BY` → sort key. The NULL rule is the surprise for most developers.

**Complexity:** O(n + g log g).

**Common mistakes**

- Treating NULL status as 'not CANCELLED' and keeping the row (Python's `None != 'CANCELLED'` is True!).

## Exercise 2 — 🟡 Medium · The same query in pandas

**Working solution**

In [ ]:
import pandas as pd

def country_report_pd(df):
    valid = df[df["status"].notna() & (df["status"] != "CANCELLED")]
    out = valid.groupby("country", as_index=False).agg(n=("amount", "size"), total=("amount", "sum"))
    out["total"] = out["total"].round(2)
    return out[out["n"] >= 2].sort_values(["total", "country"], ascending=[False, True]).reset_index(drop=True)

**Explanation**

pandas' `!=` treats NaN as 'not equal' (True) — the opposite of SQL — so NULL handling must be explicit. Spark follows SQL: `F.col('status') != 'CANCELLED'` drops nulls.

**Complexity:** O(n).

**Common mistakes**

- `df[df.status != 'CANCELLED']` keeps NaN statuses.

## Exercise 3 — 🟡 Medium · Three-valued logic and NULL-aware aggregates

**Working solution**

In [ ]:
def sql_eq(a, b):
    if a is None or b is None:
        return None
    return a == b

def sql_and(a, b):
    if a is False or b is False:
        return False
    if a is None or b is None:
        return None
    return True

def sql_or(a, b):
    if a is True or b is True:
        return True
    if a is None or b is None:
        return None
    return False

def sql_count(values, star=False):
    return len(values) if star else sum(1 for v in values if v is not None)

def sql_avg(values):
    present = [v for v in values if v is not None]
    return sum(present) / len(present) if present else None

**Explanation**

`WHERE` keeps only TRUE (not UNKNOWN), `NULL = NULL` is UNKNOWN (use `IS NULL` / Spark's null-safe `<=>` / `eqNullSafe`), and aggregates skip NULLs. These rules explain
most 'missing rows' bugs in joins and filters.

**Complexity:** O(1) / O(n).

**Common mistakes**

- `WHERE col = NULL` (never matches).
- Joining on nullable keys and expecting NULLs to match.

## Exercise 4 — 🟠 Hard · Partition pruning

**Working solution**

In [ ]:
import operator

_OPS = {"=": operator.eq, "!=": operator.ne, ">=": operator.ge, "<=": operator.le, ">": operator.gt, "<": operator.lt,
        "in": lambda a, b: a in b}

def prune(paths, filters):
    for op, _ in filters.values():
        if op not in _OPS:
            raise ValueError(f"unsupported operator {op!r}")
    keep = []
    for path in paths:
        parts = dict(seg.split("=", 1) for seg in path.split("/") if "=" in seg)
        if all(col not in parts or _OPS[op](parts[col], value) for col, (op, value) in filters.items()):
            keep.append(path)
    return keep, len(paths) - len(keep)

**Explanation**

Partition columns live in the *path*, so a filter on them can skip whole files before reading a byte — Spark shows this as `PartitionFilters` in `explain()`. Filters on regular
columns can only be pushed into file formats with statistics (Parquet row-group min/max: `PushedFilters`). Choose partition columns you filter on often, with moderate cardinality.

**Complexity:** O(files × filters).

**Common mistakes**

- Partitioning by a high-cardinality column (customer_id) → millions of tiny files.
- Filtering on a *derived* expression of the partition column (`year(dt) = 2026`) which may disable pruning.

## Challenge — 🔴 Very Hard · A lazy DataFrame with pushdown and explain()

**Working solution**

In [ ]:
import csv
import operator

_LAZY_OPS = {"=": operator.eq, "!=": operator.ne, ">": operator.gt, ">=": operator.ge, "<": operator.lt, "<=": operator.le}

class LazyFrame:
    def __init__(self, path, plan=()):
        self.path = path
        self.plan = tuple(plan)
        self.last_scan_stats = None

    @classmethod
    def read_csv(cls, path):
        return cls(path)

    def _add(self, node):
        return LazyFrame(self.path, self.plan + (node,))

    def filter(self, column, op, value):
        if op not in _LAZY_OPS:
            raise ValueError(f"unsupported operator {op}")
        return self._add(("filter", column, op, value))

    def with_column(self, name, func, depends_on):
        return self._add(("with_column", name, func, tuple(depends_on)))

    def select(self, *cols):
        return self._add(("select", cols))

    # ---------------- optimizer ----------------
    def _optimize(self):
        derived = {n[1] for n in self.plan if n[0] == "with_column"}
        scan_filters = [n for n in self.plan if n[0] == "filter" and n[1] not in derived]
        rest = [n for n in self.plan if not (n[0] == "filter" and n[1] not in derived)]
        selects = [n for n in rest if n[0] == "select"]
        final_cols = set(selects[-1][1]) if selects else None
        needed = set()
        if final_cols is not None:
            wanted = set(final_cols) | {n[1] for n in rest if n[0] == "filter"}
            for node in reversed(rest):
                if node[0] == "with_column" and node[1] in wanted:
                    wanted |= set(node[3])
            needed = {c for c in wanted if c not in derived}
        needed |= {f[1] for f in scan_filters}
        return scan_filters, rest, (needed if final_cols is not None else None)

    def explain(self):
        scan_filters, rest, needed = self._optimize()
        cols = "*" if needed is None else ", ".join(sorted(needed))
        lines = [f"Scan csv [{cols}] filters=[{', '.join(f'{c} {op} {v!r}' for _, c, op, v in scan_filters)}]"]
        for node in rest:
            if node[0] == "filter":
                lines.append(f"Filter {node[1]} {node[2]} {node[3]!r}")
            elif node[0] == "with_column":
                lines.append(f"WithColumn {node[1]} <- {list(node[3])}")
            else:
                lines.append(f"Select {list(node[1])}")
        return "\n".join(lines)

    @staticmethod
    def _test(row, column, op, value):
        raw = row.get(column)
        if raw is None or raw == "":
            return False
        try:
            return _LAZY_OPS[op](type(value)(raw) if not isinstance(raw, type(value)) else raw, value)
        except (TypeError, ValueError):
            return False

    def collect(self):
        scan_filters, rest, needed = self._optimize()
        rows, read, kept = [], 0, 0
        with open(self.path, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            columns = reader.fieldnames if needed is None else [c for c in reader.fieldnames if c in needed]
            for raw in reader:
                read += 1
                if all(self._test(raw, c, op, v) for _, c, op, v in scan_filters):
                    kept += 1
                    rows.append({c: raw[c] for c in columns})
        self.last_scan_stats = {"rows_read": read, "rows_after_scan_filters": kept, "columns_read": len(columns)}
        for node in rest:
            if node[0] == "filter":
                _, c, op, v = node
                rows = [r for r in rows if self._test(r, c, op, v)]
            elif node[0] == "with_column":
                _, name, func, _deps = node
                rows = [{**r, name: func(r)} for r in rows]
            else:
                rows = [{c: r.get(c) for c in node[1]} for r in rows]
        return rows

**Explanation**

Because the plan is data, an optimizer can rewrite it before running: moving filters next to the scan means fewer rows flow through later steps; computing needed columns means fewer bytes
are read (with Parquet, whole columns are skipped on disk). This is what Spark's Catalyst optimizer does — `explain(True)` shows the parsed, analyzed, optimized and physical plans.

**Complexity:** O(rows × needed columns).

**Common mistakes**

- Executing eagerly inside `filter`.
- Pushing a filter on a *derived* column below the step that creates it.